In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

Random Walk Monte Carlo (RWMC)

In [ ]:
def RWMCMC(q0,dim,pdf,sig,L, prtAcpRatio = False):
    """
        RWMC for the proposal kernel
            Q(x, dy) = q(x,y)dy = z^{-1} exp( - (2*sig^{-2}) (x-y)^2)dy
        INPUTS:
            q0 -initial state
            dim- target dimention
            pdf- target density
            sig- Diagonal gaussian proposal kernel variance
            L - Number of samples drawn
    """
    samID = 0
    acp = 0
    samp = []
    qc = q0
    qp = qc
    while(samID < L):
        samID = samID + 1
        samp.append(qc)
        qp = np.random.normal(qc,sig,dim)
        a = pdf(qp) / pdf(qc)
        u = np.random.uniform()
        if(u <= a):
            qc = qp
            acp = acp + 1 
    if prtAcpRatio:
        ar = acp/L
        print("RWMC AR:" + str(ar) + " ", end = '')
    return samp

Hamiltonian Monte Carlo (HMC)

In [3]:
#First Set up the Standard Leap-Frog (Verlet) numerical integration scheme

def LeapInt(q0, p0, m,f,dt,T):
    """Leap Frog Integration of a system of the form
        m dq/dt = p,      q(0) =q0
        dp/dt = -f(p)   p(0) =p0
        INPUTS: q0, p0: Initial conditions
                f: gradiant of the potential
                m: mass
                dt: time step size
                T: integration time
    """
    tc = 0
    qc = np.array(q0)
    pc = np.array(p0)
    while tc < T:
        pc= pc-f(qc)*.5*dt
        qc= qc+ m**(-1) * pc*dt
        pc= pc-f(qc)*.5*dt
        tc = tc +dt  
    return qc, pc
    

def HMC(q0,dU,dim,Ham,m,dt,Tint,L, prtAcpRatio = False):
    samID = 0
    acp = 0 #Number of acceptances
    samp = []
    qc = np.array(q0)
    pc = np.random.normal(0,m**.5,dim)
    qprop = qc
    pprop = pc
    while(samID < L):
        samID = samID + 1
        samp.append(qc)
        pc = np.random.normal(0,m**.5,dim)
        qprop,pprop = LeapInt(qc, pc,m,dU,dt,Tint)
        a = exp(Ham(qc,pc) - Ham(qprop,pprop))
        u = np.random.uniform()
        if(u < a):
            qc = qprop
            acp = acp + 1
    if prtAcpRatio:
        ar = acp/L
        print("HMC AR:" + str(ar) + " ", end = '')
    return samp

Vanilla Multiproposal Sampliers

In [ ]:
from numpy.random import multivariate_normal as mvnormal


def MultPropGauss(xinit,pdf,sig,NBB,L):
    samID = 0
    acp = 0
    covM = np.diag(sig)
    samp = []
    xc = xinit
    while samID < L:
        samID = samID + 1
        samp.append(xc)
        tjCentx = mvnormal(xc,covM)
        xprops = np.append([xc], mvnormal(tjCentx,covM,NBB), axis=0)
        aprobsAB = [pdf(cprop) for cprop in xprops]
        NMpb = sum(aprobsAB)**(-1)
        aprobs = [NMpb*cur for cur in aprobsAB]
        chIndx = np.random.choice(len(xprops),1,p=aprobs)
        xc = xprops[chIndx[0]]
    return samp

The Vanilla PcN Algorithm

In [4]:
from math import exp 
import numpy as np

def pCNProp(q,dim,Cov,rho):
    return rho*q + (1 - rho**2)**.5*np.random.multivariate_normal(np.zeros(dim),Cov)

def pCN(q0,dim,Cov,rho,Pot,L, prtAcpRatio = False):
    """pCN Algorithm to sample from of measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        q0 - inital state
        Cov- Covariance of mu0
        rho-pCN parameter between [0,1] (rho := (4 − delta)(4 + delta)^{-1} where delta is the time step size)
        From the state q make the proposal
            qprop = rho * q + sqrt(1- rho**2) * v
        where v ~ N(0,Cov).
        Accept qprop with probability
            alpha(q,qprop) = 1 wedge exp(Pot(q)-Pot(prop))
    """
    if prtAcpRatio:
        samID = 0
        acp = 0 #Number of acceptances
        samp = []
        qc = np.array(q0)
        qprop = qc
        while(samID < L):
            samID = samID + 1
            samp.append(qc)
            qprop = pCNProp(qc,dim,Cov,rho)
            a = exp(Pot(qc) - Pot(qprop))
            u = np.random.uniform()
            if(u < a):
                qc = qprop
                acp = acp + 1
        ar = acp/L
        print("PcN AR:" + str(ar) + " ", end = '')
    else: 
        samID = 0
        samp = []
        qc = np.array(q0)
        qprop = qc
        while(samID < L):
            samID = samID + 1
            samp.append(qc)
            qprop = pCNProp(qc,dim,Cov,rho)
            a = exp(Pot(qc) - Pot(qprop))
            u = np.random.uniform()
            if(u < a):
                qc = qprop
    return samp 

def pCNrhoFinder(q0, dim, Cov, rhomin, rhomax, rhoInc, arMin, arMax, Pot, L):
    rhoCur = rhomin
    ar = 0
    while rhoCur < rhomax: 
        samID = 0
        acp = 0 #Number of acceptances
        qc = np.array(q0)
        qprop = qc
        while(samID < L):
            samID = samID + 1
            qprop = pCNProp(qc,dim,Cov,rhoCur)
            a = exp(Pot(qc) - Pot(qprop))
            u = np.random.uniform()
            if(u < a):
                qc = qprop
                acp = acp + 1
        ar = acp/L
        if ar > arMin and ar < arMax:
            print("AC found: " + str(ar))
            return rhoCur
        rhoCur = rhoCur + rhoInc
    print("AC found: " + str(ar))
    print("WARNING: No rho found")
    return rhoCur
    

Multiproposal PcN Algorithm and Multiple Parallel Chains of PcN

In [1]:
from math import exp 
import numpy as np
from concurrent.futures import ThreadPoolExecutor

def MpCN(q0,dim,Cov,rho,Pot,NProps,L):
    """
    The Multiproposal PcN Algorithm 
        Provdies to samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
    """
    rng = np.random.default_rng()
    samp = np.empty((L + 1, dim), dtype=float) #Make an array for the samples
    samp[0] = np.array(q0)
    Cov_chol = np.linalg.cholesky(Cov) #Find E such that EE^* = C
    eta = np.sqrt(1.0 - rho * rho)
    for samID in range(1, L + 1):
        qtjCen = rho * samp[samID -1] + eta * Cov_chol @ rng.standard_normal(dim) #draw initial center point
        curProps = np.concatenate((samp[samID -1][:,None],rho* qtjCen[..., None] + eta * Cov_chol @ rng.standard_normal((dim,NProps))),axis =-1).T #cloud of proposals
        logAcp = np.empty(NProps + 1, dtype=float)
        for j in range(NProps+1):
            logAcp[j] = -1*Pot(curProps[j])
        logAcp_max = np.max(logAcp)
        Acp = np.exp(logAcp - logAcp_max)  # stabilised weights
        idx = rng.choice(NProps+1, p=Acp / Acp.sum())
        samp[samID] = curProps[idx].copy()
    return samp 


def locMpCN(q0,dim,Cov,rho,Pot,NProps,L):
    """
    The `local Multiproposal PcN Algorithm'
        Samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
    """
    rng = np.random.default_rng()
    samp = np.empty((L + 1, dim), dtype=float) #Make an array for the samples
    samp[0] = np.array(q0)
    Cov_chol = np.linalg.cholesky(Cov) #Find E such that EE^* = C
    eta = np.sqrt(1.0 - rho * rho)
    potpreFac = rho*(1.0 - rho)*(1.0 - rho * rho)**(-1) #Scaling prefactor in the potential
    rhoinv = rho**(-1)
    for samID in range(1, L + 1):
        curProps = (rho* samp[samID -1][:,None] + eta * Cov_chol @ rng.standard_normal((dim,NProps))).T 
        #cloud of proposals
        logAcp = np.empty(NProps, dtype=float)
        for j in range(NProps):
            logAcp[j] = -1* potpreFac*Pot(rhoinv*curProps[j])
        logAcp_max = np.max(logAcp)
        Acp = np.exp(logAcp - logAcp_max)  # stabilised weights
        idx = rng.choice(NProps, p=Acp / Acp.sum())
        samp[samID] = curProps[idx].copy()
    return samp 


def locMpCNMTM(q0,dim,Cov,rho,Pot,NProps,L,PrintAcpRate = False):
    """
    The `local Multiproposal PcN Algorithm' with the MTM correction
        Samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
    """
    nmRjt = 0.0
    rng = np.random.default_rng()
    samp = np.empty((L + 1, dim), dtype=float) #Make an array for the samples
    samp[0] = np.array(q0)
    Cov_chol = np.linalg.cholesky(Cov) 
    #Find E such that EE^* = C
    eta = np.sqrt(1.0 - rho * rho)
    #Scaling prefactor in the potential
    potpreFac = rho*(1.0 - rho)*(1.0 - rho * rho)**(-1) 
    rhoinv = rho**(-1)
    for samID in range(1, L + 1):
        prevSam = samp[samID -1].copy()
        curProps = (rho* prevSam[:,None]  + eta * Cov_chol @ rng.standard_normal((dim,NProps))).T 
        #the main cloud of proposals
        logAcp = np.empty(NProps, dtype=float)
        #log acceptance probabilities
        for j in range(NProps):
            logAcp[j] = -1* potpreFac*Pot(rhoinv*curProps[j])
        logAcp_max = np.max(logAcp)
        Acp = np.exp(logAcp - logAcp_max)  # stabilised weights
        Acp_norm = Acp.sum()
        idx = rng.choice(NProps, p=Acp / Acp_norm)
        mtmProp = curProps[idx].copy()

        #MTM Backward Step
        
        mtmPropsAux = (rho* mtmProp[:,None] + eta * Cov_chol @ rng.standard_normal((dim,NProps -1))).T 
        #MTM Reverse Proproposal
        logAcpDenomMTM = np.empty(NProps, dtype=float)
        logAcpDenomMTM[0] = -1*potpreFac*Pot(rhoinv*prevSam)
        for j in range(NProps -1):
            logAcpDenomMTM[j+1] = -1* potpreFac*Pot(rhoinv*mtmPropsAux[j])
        logAcpDen_max = np.max(logAcpDenomMTM)
        #Regular Acceptance
        #MTMacpNum =exp(-1*Pot(mtmProp))*exp(-1*potpreFac* Pot(rhoinv*prevSam) -logAcpDen_max)* Acp_norm
        #MTMacpDen =exp(-1*Pot(prevSam))*exp(-1*potpreFac* Pot(rhoinv*mtmProp) -logAcp_max) *np.exp(logAcpDenomMTM - logAcpDen_max).sum()
        #MTMacp = min(1, MTMacpNum/MTMacpDen)

        
        #log acceptance
        logMTMacpNum = -1*Pot(mtmProp) -potpreFac* Pot(rhoinv*prevSam) + np.log(Acp_norm) + logAcp_max
        logMTMacpDen = -1*Pot(prevSam) - potpreFac* Pot(rhoinv*mtmProp) + np.log(np.exp(logAcpDenomMTM - logAcpDen_max).sum()) + logAcpDen_max

        logMTMacp = min(0, logMTMacpNum -logMTMacpDen)
        MTMacp = np.exp(logMTMacp)
        
        if rng.random() < MTMacp:
            samp[samID] = mtmProp
        else:
            nmRjt += 1
            samp[samID] = prevSam

    if PrintAcpRate:
        print("Rejection rate:")
        print(nmRjt/L)
        
    return samp

def MpCNBBMTM(q0,dim,Cov,rho,Pot,NProps,L,PrintAcpRate = False):
    """
    The `bubble bath' Multiproposal PcN Algorithm with MTM correction.
        Samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
    """
    nmRjt = 0.0
    rng = np.random.default_rng()
    samp = np.empty((L + 1, dim), dtype=float) #Make an array for the samples
    samp[0] = np.array(q0)
    Cov_chol = np.linalg.cholesky(Cov) 
    #Find E such that EE^* = C
    eta = np.sqrt(1.0 - rho * rho)
    for samID in range(1, L + 1):
        curProps = (rho* samp[samID -1][:,None]  + eta * Cov_chol @ rng.standard_normal((dim,NProps))).T 
        #the main cloud of proposals
        logAcp = np.empty(NProps, dtype=float)
        #log acceptance probabilities
        for j in range(NProps):
            logAcp[j] = -1* Pot(curProps[j])
        logAcp_max = np.max(logAcp)
        Acp = np.exp(logAcp - logAcp_max)  # stabilised weights
        Acp_norm = Acp.sum()
        idx = rng.choice(NProps, p=Acp / Acp_norm)
        mtmProp = curProps[idx].copy()

        #MTM Backward Step
        
        mtmPropsAux = (rho* mtmProp[:,None] + eta * Cov_chol @ rng.standard_normal((dim,NProps -1))).T 
        #MTM Reverse Proproposal
        logAcpDenomMTM = np.empty(NProps, dtype=float)
        prevSam = samp[samID -1].copy()
        logAcpDenomMTM[0] = -1*Pot(prevSam)
        for j in range(NProps -1):
            logAcpDenomMTM[j+1] = -1* Pot(mtmPropsAux[j])
        logAcpDen_max = np.max(logAcpDenomMTM)
        #Normal Acceptance            
        #MTMacpNum = np.exp( logAcp_max- logAcpDen_max)*Acp_norm
        #MTMacpDen = np.exp(logAcpDenomMTM - logAcpDen_max).sum()
        #MTMacp = min(1, MTMacpNum/MTMacpDen)

        #Log Acceptance
        logMTMacpNum = np.log(Acp_norm) +logAcp_max
        logMTMacpDen = np.log(np.exp(logAcpDenomMTM - logAcpDen_max).sum()) + logAcpDen_max
        
        logMTMacp = min(0, logMTMacpNum -logMTMacpDen)
        MTMacp = np.exp(logMTMacp)
        
        if rng.random() < MTMacp:
            samp[samID] = mtmProp
        else:
            nmRjt += 1
            samp[samID] = prevSam
    if PrintAcpRate:
        print("Rejection rate:")
        print(nmRjt/L)
    return samp

def MpCNBBMTM_Bizaro(q0,dim,Cov,rho,Pot,NProps,L):
    """
    The `bubble bath' Multiproposal PcN Algorithm with MTM correction.
    Saving a strange bug version where the proped point is also in the denominator.
        Samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
    """
    #nmRjt = 0.0
    rng = np.random.default_rng()
    samp = np.empty((L + 1, dim), dtype=float) #Make an array for the samples
    samp[0] = np.array(q0)
    Cov_chol = np.linalg.cholesky(Cov) 
    #Find E such that EE^* = C
    eta = np.sqrt(1.0 - rho * rho)
    for samID in range(1, L + 1):
        prevSam = samp[samID -1][:,None]
        curProps = (rho* prevSam  + eta * Cov_chol @ rng.standard_normal((dim,NProps))).T 
        #the main cloud of proposals
        logAcp = np.empty(NProps, dtype=float)
        #log acceptance probabilities
        for j in range(NProps):
            logAcp[j] = -1* Pot(curProps[j])
        logAcp_max = np.max(logAcp)
        Acp = np.exp(logAcp - logAcp_max)  # stabilised weights
        Acp_norm = Acp.sum()
        idx = rng.choice(NProps, p=Acp / Acp_norm)
        mtmProp = curProps[idx].copy()

        #MTM Backward Step
        #UNDER CONSTRUCTION
        #lognumprefac = -1*(Pot(mtmProp) + potpreFac* Pot(rhoinv*prevSam))
        #logdenomprefac = -1*(Pot(prevSam) + potpreFac* Pot(rhoinv*mtmProp))
        
        mtmPropsAux = (rho* mtmProp[:,None] + eta * Cov_chol @ rng.standard_normal((dim,NProps -1))).T 
        #MTM Reverse Proproposal
        logAcpDenomMTM = np.empty(NProps, dtype=float)
        logAcpDenomMTM[0] = -1*Pot(mtmProp)
        for j in range(NProps -1):
            logAcpDenomMTM[j+1] = -1* Pot(mtmPropsAux[j])
        logAcpDen_max = np.max(logAcpDenomMTM)
        MTMacpNum = exp( logAcp_max- logAcpDen_max)*Acp_norm
        MTMacpDen = np.exp(logAcpDenomMTM - logAcpDen_max).sum()
        MTMacp = min(1, MTMacpNum/MTMacpDen)
        
        if rng.random() < MTMacp:
            samp[samID] = mtmProp
        else:
            #nmRjt += 1
            samp[samID] = samp[samID -1].copy()

    #print("rejection rate:")
    #print(nmRjt/L)
    return samp

    
def MpCN_parPot(q0,dim,Cov,rho,Pot,NProps,L):
    """
    The Multiproposal PcN Algorithm 
        Provdies to samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
        Note: Attempt to parallelize the potential evaluation...
    """
    rng = np.random.default_rng()
    samp = np.empty((L + 1, dim), dtype=float) #Make an array to put the samples in
    samp[0] = np.array(q0)
    Cov_chol = np.linalg.cholesky(Cov) #Find L such that LL^* = C
    eta = np.sqrt(1.0 - rho * rho)
    _pool = ThreadPoolExecutor() #set-up for parallel execution of Pot evaluations
    mPot = lambda x : -1 * Pot(x)
    for samID in range(1, L + 1):
        qtjCen = rho * samp[samID -1] + eta * Cov_chol @ rng.standard_normal(dim) #draw initial center point
        curProps = np.concatenate((samp[samID -1][:,None],rho* qtjCen[..., None] + eta * Cov_chol @ rng.standard_normal((dim,NProps))),axis =-1).T #cloud of proposals
        logAcp = np.array(list(_pool.map(mPot, curProps)), dtype=float)
        logAcp_max = np.max(logAcp)
        Acp = np.exp(logAcp - logAcp_max)  # stabilised weights
        idx = rng.choice(NProps+1, p=Acp / Acp.sum())
        samp[samID] = curProps[idx].copy()
    return samp   

def MpCN_Olde_Ver(q0,dim,Cov,rho,Pot,NProps,L):
    """
    The Multiproposal PcN Algorithm 
        Provdies to samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
    NOTE: OLDE (but working) unoptimized version
    """
    samID = 0
    samp = []
    qc = np.array(q0)
    while(samID < L):
        samID = samID + 1
        samp.append(qc)
        qtjCen = pCNProp(qc,dim,Cov,rho)
        curProps = [qc]
        curAcp = [exp(-1*Pot(qc))]
        for j in range(NProps):
            qj = pCNProp(qtjCen,dim,Cov,rho)
            curProps.append(qj)
            curAcp.append(exp(-1*Pot(qj)))
        NMpb = sum(curAcp)**(-1)
        aprobs = [NMpb*cur for cur in curAcp]
        chIndx = np.random.choice(len(curProps),1,p=aprobs)
        qc = curProps[chIndx[0]]
    return samp



def MpCNBB(q0,dim,Cov,rho,Pot,NProps,L):
    """
    The Multiproposal PcN Algorithm *without Tj Correction*
    """
    samID = 0
    samp = []
    qc = np.array(q0)
    while(samID < L):
        samID = samID + 1
        samp.append(qc)
        curProps = [qc]
        curAcp = [exp(-1*Pot(qc))]
        for j in range(NProps):
            qj = pCNProp(qc,dim,Cov,rho)
            curProps.append(qj)
            curAcp.append(exp(-1*Pot(qj)))
        NMpb = sum(curAcp)**(-1)
        aprobs = [NMpb*cur for cur in curAcp]
        chIndx = np.random.choice(len(curProps),1,p=aprobs)
        qc = curProps[chIndx[0]]
    return samp

def MpCNMSBB(q0,dim,Cov,rhoLst,Pot,NPropsLst,L):
    """
    The Multiproposal Multiscale PcN Algorithm *without Tj Correction*
    """
    samID = 0
    samp = []
    qc = np.array(q0)
    while(samID < L):
        samID = samID + 1
        samp.append(qc)
        curProps = [qc]
        curAcp = [exp(-1*Pot(qc))]
        for rhoCur in rhoLst:
            for NProps in NPropsLst:
                for j in range(NProps):
                    qj = pCNProp(qc,dim,Cov,rhoCur)
                    curProps.append(qj)
                    curAcp.append(exp(-1*Pot(qj)))
        NMpb = sum(curAcp)**(-1)
        aprobs = [NMpb*cur for cur in curAcp]
        chIndx = np.random.choice(len(curProps),1,p=aprobs)
        qc = curProps[chIndx[0]]
    return samp

def pCNStp(qc,dim,Cov,rho,Pot):
        qprop = pCNProp(qc,dim,Cov,rho)
        a = exp(Pot(qc) - Pot(qprop))
        u = np.random.uniform()
        if(u < a):
            return qprop
        else:
            return qc

def parChnpCN(nChain,dim,Cov, rho, Pot, L):
    samID = 0
    samp = []
    qcLst = [np.random.multivariate_normal(np.zeros(dim),Cov) for j in range(nChain)]
    while(samID < L):
        curChID = 0
        while(curChID < nChain):
            samID = samID + 1
            samp.append(qcLst[curChID])
            curChID += 1 
            qcLst = [pCNStp(qc,dim,Cov,rho,Pot) for qc in qcLst]
    return samp

def parChnpCNIC(q0, nChain,dim,Cov, rho, Pot, L):
    samID = 0
    samp = []
    qcLst = [q0 for j in range(nChain)]
    while(samID < L):
        curChID = 0
        while(curChID < nChain):
            samID = samID + 1
            samp.append(qcLst[curChID])
            curChID += 1 
            qcLst = [pCNStp(qc,dim,Cov,rho,Pot) for qc in qcLst]
    return samp

Some Plotting Utilities

####UNDER CONSTRUCTION#####

from math import exp
import numpy as np
#import cupy as cp


def MpCN_GPU(q0,dim,Cov,rho,Pot,NProps,L):
    """
    The Multiproposal PcN Algorithm 
    Written for GPU Use
        Provdies to samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
    """
    rng = cp.random.default_rng()
    samp = cp.empty((L + 1, dim), dtype=float) 
    #Make an array for the samples
    samp[0] = cp.array(q0)
    Cov_chol = cp.linalg.cholesky(Cov) 
    #Find E such that EE^* = C
    eta = np.sqrt(1.0 - rho * rho)
    for samID in range(1, L + 1):
        qtjCen = rho * samp[samID -1] + eta * Cov_chol @ rng.standard_normal(dim) 
        #draw initial center point
        curProps = cp.concatenate((samp[samID -1][:,None],rho* qtjCen[..., None] + eta * Cov_chol @ rng.standard_normal((dim,NProps))),axis =-1).T #cloud of proposals
        logAcp = cp.empty(NProps + 1, dtype=float)
        for j in range(NProps+1):
            logAcp[j] = -1*Pot(curProps[j])
        logAcp_max = cp.max(logAcp)
        Acp = cp.exp(logAcp - logAcp_max)  # stabilised weights
        idx = rng.choice(NProps+1, p=Acp / Acp.sum())
        samp[samID] = curProps[idx].copy()
    return cp.asnumpy(samp) 

In [1]:
import matplotlib.pyplot as plt
import numpy as np

def getComp(sampLst,indx):
    return [item[indx] for item in sampLst]

def sampMRunAve(sampList,momF,stV,skpL=1):
    """
    Given the time series {X_1,... ,X_N} and the function F
    computes the sample average time series
    """
    smpM = [momF(x) for x in sampList]
    smpRA = []
    k = stV
    LnSm = len(smpM)
    while k < LnSm:
        smpRA.append(sum(smpM[:k]) / k)
        k = k + skpL
    return smpRA

def compACSampFn(sampLst,Fn,stVal, N,trMu,trSigSq, MaxLag):
    """
    Approximates the AutoCorelation
        hrho_t = frac{1}{N sigma^2} sum_{l = s}^{s+N} (Fn(X_{l+t}) - mu)(Fn(X_l) - mu)
    """
    sampLstF = [Fn(smp) for smp in sampLst]
    acLst = []
    curLag = 1
    
    while curLag < MaxLag:
        lgSum = 0
        for l in range(stVal,N+stVal):
            lgSum =lgSum + (sampLstF[l+ curLag] - trMu)*(sampLstF[l] - trMu)
        currAC = lgSum/(N*trSigSq)
        acLst.append(currAC)
        curLag = curLag + 1
    return acLst

def compMoment(momf, smpList):
    smpM = [momf(x) for x in smpList]
    N = len(smpList)
    return N**(-1) *sum(smpM)

def plt2DHeat(ran,dr,fn):
    xaxi= np.arange(-ran, ran, dr)
    yaxi= np.arange(-ran, ran, dr)
    Z = np.zeros((len(xaxi), len(yaxi)))
    for i in range(len(xaxi)):
        for j in range(len(yaxi)):
            Z[i][j] = fn([xaxi[i],yaxi[j]])    
    plt.contourf(xaxi,yaxi, Z, 70, cmap='RdGy')
        
def plt2DCont(ran,dr,fn,l):
    xaxi= np.arange(-ran, ran, dr)
    yaxi= np.arange(-ran, ran, dr)
    Z = np.zeros((len(xaxi), len(yaxi)))
    for i in range(len(xaxi)):
        for j in range(len(yaxi)):
            Z[i][j] = fn([xaxi[i],yaxi[j]])
    plt.contour(xaxi,yaxi, Z, levels=l)
    
def pltf(f,qran,dq):
    X = []
    fX = []
    qCur = -qran
    while qCur < qran:
        X.append(qCur)
        fX.append(f(qCur))
        qCur = qCur + dq
    plt.plot(X,fX)

def pltfc(f,qcent,qran,dq):
    X = []
    fX = []
    qCur = -qran + qcent
    while qCur < qran + qcent:
        X.append(qCur)
        fX.append(f(qCur))
        qCur = qCur + dq
    plt.plot(X,fX)
    
def Hist2DPlt(xsam,ysam,ran,dran):
    numbins= int(ran/dran)
    x_bins = np.linspace(-ran, ran, numbins) 
    y_bins = np.linspace(-ran, ran, numbins)
    plt.hist2d(xsam,ysam,bins= [x_bins,y_bins])
    
    
def Hist2DPlt0(xsam,ysam,xMin,xMax,yMin,yMax,dr):
    numbinsX= int((xMax-xMin)/dr)
    numbinsY= int((yMax-yMin)/dr)
    x_bins = np.linspace(xMin, xMax, numbinsX) 
    y_bins = np.linspace(yMin, yMax, numbinsY)
    plt.hist2d(xsam, ysam,bins= [x_bins,y_bins], cmap=plt.cm.jet)

def plt2DCont0(xMin,xMax,yMin,yMax,dr,fn,l):
    xaxi= np.arange(xMin, xMax, dr)
    yaxi= np.arange(yMin, yMax, dr)
    Z = np.zeros((len(yaxi), len(xaxi)))
    for i in range(len(xaxi)):
        for j in range(len(yaxi)):
            Z[j][i] = fn([xaxi[i],yaxi[j]])
    plt.contour(Z, levels=l, colors='black')

#More Utility Functions

#Graphing Utils

#Make a Histogram from a list to a file

def makeHistGrid(R, dr, sampList, NumParams,saveLoc, hidePlt = True):
    SampLstsGd = [getComp(sampList,j) for j in range(0,NumParams)]
    numbins= int(2*R/dr)
    x_bins = np.linspace(-R, R, numbins) 
    y_bins = np.linspace(-R, R, numbins)

    fig, axs = plt.subplots(NumParams, NumParams,figsize=(15,15))
    for i in range(0,NumParams):
        for j in range(0,NumParams):
            if i == j:
                axs[i,j].hist(SampLstsGd[i], density=True, bins=x_bins)
            else:
                axs[i,j].hist2d(SampLstsGd[j],SampLstsGd[i],bins= [x_bins,y_bins])
        for ax in axs.flat:
            ax.label_outer()
    plt.savefig(saveLoc)
    if hidePlt:
        plt.close()




#Make Time Series Plots

def mkTSPlt(MpCNSamp, MpCNBBSamp, DistDim, pVal, rhoVal, strVTS, endVTS, saveLoc,hidePlt = True):
    
    #plt.figure(figIndx);
    fig, axs = plt.subplots(DistDim,2, figsize=(45,45))
    
    #Generate MpCN TimeSeries
    for j in range(0,DistDim):
        X = []
        fX = []
        CurX = strVTS
        while CurX < endVTS:
            X.append(float(CurX))
            fX.append(MpCNSamp[CurX][j])
            CurX = CurX + 1
        axs[j,0].plot(X,fX);
    
    #Generate MpCNbb TimeSeries
    for j in range(0,DistDim):
        X = []
        fX = []
        CurX = strVTS
        while CurX < endVTS:
            X.append(float(CurX))
            fX.append(MpCNBBSamp[CurX][j])
            CurX = CurX + 1
        axs[j,1].plot(X,fX);

    for ax in axs.flat:
        ax.label_outer();
    smpRMpCNDtStr= "p = " + str(pVal) + ", rho =" + str(rhoVal)
    fig.suptitle("Timeseries of "+ smpRMpCNDtStr, fontsize='xx-large')
    plt.savefig(saveLoc)
    if hidePlt:
        plt.close()
        
def mkTSPltEx(MpCNSamp, MpCNBBSamp, tradpCNSamp, DistDim, pVal, rhoVal, strVTS, endVTS, saveLoc,hidePlt = True):
    
    #plt.figure(figIndx);
    fig, axs = plt.subplots(DistDim,3, figsize=(45,45))
    
    #Generate MpCN TimeSeries
    for j in range(0,DistDim):
        X = []
        fX = []
        CurX = strVTS
        while CurX < endVTS:
            X.append(float(CurX))
            fX.append(MpCNSamp[CurX][j])
            CurX = CurX + 1
        axs[j,0].plot(X,fX);
    
    #Generate MpCNbb TimeSeries
    for j in range(0,DistDim):
        X = []
        fX = []
        CurX = strVTS
        while CurX < endVTS:
            X.append(float(CurX))
            fX.append(MpCNBBSamp[CurX][j])
            CurX = CurX + 1
        axs[j,1].plot(X,fX);
        
    # Generate pCNbb TimeSeries
    for j in range(0,DistDim):
        X = []
        fX = []
        CurX = strVTS
        while CurX < endVTS:
            X.append(float(CurX))
            fX.append(tradpCNSamp[CurX][j])
            CurX = CurX + 1
        axs[j,2].plot(X,fX);

    for ax in axs.flat:
        ax.label_outer();
    smpRMpCNDtStr= "p = " + str(pVal) + ", rho =" + str(rhoVal)
    fig.suptitle("Timeseries of "+ smpRMpCNDtStr, fontsize='xx-large')
    plt.savefig(saveLoc)
    if hidePlt:
        plt.close()

#Function to compute the auto-correlation
def compACSamp(sampLst,stVal, N,trMu,trSigSq, MaxLag):
    """
    Approximates the AutoCorelation
        hrho_t = frac{1}{N sigma^2} sum_{l = s}^{s+N} (X_{l+t} - mu)(X_l - mu)
    """
    acLst = []
    curLag = 1
    
    while curLag < MaxLag:
        lgSum = 0
        for l in range(stVal,N+stVal):
            lgSum =lgSum + (sampLst[l+ curLag] - trMu)*(sampLst[l] - trMu)
        currAC = lgSum/(N*trSigSq)
        acLst.append(currAC)
        curLag = curLag + 1
    return acLst

def sampRunAve(sampList,stV,lnSA,trV, skip = 1):
    """
    Given the time series {X_1,... ,X_N} and the function F
    computes the sample average time series
    """
    smpRA = []
    sampId = []
    k = stV
    while k < lnSA:
        smpRA.append((sum(sampList[:k]) / k) - trV)
        sampId.append(k)
        k = k + skip
    return sampId,smpRA

Some Utilities to read/write to a .csv file

In [ ]:
import pandas as pd

def writeCSV(filenm, sampArray, newFile = False):
    if newFile:
        pd.DataFrame(sampArray).to_csv(filenm, mode='w', index=False, header=False)
    else:
        pd.DataFrame(sampArray).to_csv(filenm, mode='a', index=False, header=False)
    
def readCSV(filenm):   
    df = pd.read_csv(filenm)
    MpCNsamp2 = df.to_numpy()
    return MpCNsamp2

def readCSVN(filenm, NumEle):   
    df = pd.read_csv(filenm, nrows= NumEle)
    MpCNsamp2 = df.to_numpy()
    return MpCNsamp2

